In [ ]:
import pandas as pd

# Load dataset
deliveries = pd.read_csv("data/deliveries.csv")
matches = pd.read_csv("data/matches.csv")


# Show first few rows
deliveries.head()
matches.head()

# Check columns
deliveries.columns
matches.columns



In [ ]:
# merge data set
df = deliveries.merge(matches, left_on='match_id', right_on='id')

df.head()

In [ ]:
# Total runs scored by each player
player_runs = df.groupby('batter')['batsman_runs'].sum().reset_index()
player_runs.columns = ['player', 'total_runs']

player_runs.head()

In [ ]:
# Number of matches played by each player
matches_played = df.groupby('batter')['match_id'].nunique().reset_index()
matches_played.columns = ['player', 'matches']

matches_played.head()

In [ ]:
# Combine runs + matches
player_stats = player_runs.merge(matches_played, on='player')

player_stats.head()

In [ ]:
# Calculate average runs
player_stats['avg_runs'] = player_stats['total_runs'] / player_stats['matches']

player_stats.head()

In [ ]:
# Get wickets (dismissals)
wickets = df[df['dismissal_kind'].notnull()]

wickets.head()

In [ ]:
# Count wickets per bowler
bowler_wickets = wickets.groupby('bowler').size().reset_index(name='wickets')

bowler_wickets.head()

In [ ]:
# Merge wickets with player stats
player_stats = player_stats.merge(
    bowler_wickets,
    left_on='player',
    right_on='bowler',
    how='left'
)

# Fill missing values
player_stats['wickets'] = player_stats['wickets'].fillna(0)

player_stats.head()

In [ ]:
# Drop duplicate column
player_stats = player_stats.drop(columns=['bowler'])

player_stats.head()

In [ ]:
# Create performance score
player_stats['performance'] = (
    player_stats['total_runs'] + (player_stats['wickets'] * 20)
)

player_stats.head()

In [ ]:
def get_role(row):
    if row['wickets'] > 50 and row['total_runs'] > 1000:
        return 'All-Rounder'
    elif row['wickets'] > 50:
        return 'Bowler'
    else:
        return 'Batsman'

player_stats['role'] = player_stats.apply(get_role, axis=1)

player_stats.head()

In [ ]:
# Runs scored by player at each venue
venue_runs = df.groupby(['batter', 'venue'])['batsman_runs'].sum().reset_index()

venue_runs.head()

In [ ]:
venue_matches = df.groupby(['batter', 'venue'])['match_id'].nunique().reset_index()
venue_matches.columns = ['batter', 'venue', 'matches']

venue_matches.head()

In [ ]:
venue_stats = venue_runs.merge(venue_matches, on=['batter', 'venue'])

venue_stats.head()

In [ ]:
venue_stats['avg_runs'] = venue_stats['batsman_runs'] / venue_stats['matches']

venue_stats.head()

In [ ]:
# venue
selected_venue = "Wankhede Stadium"
venue_filtered = venue_stats[venue_stats['venue'] == selected_venue]
final_df = venue_filtered.merge(player_stats, left_on='batter', right_on='player')
final_df = final_df.sort_values(by='performance', ascending=False)

In [ ]:
batsmen = final_df[final_df['role'] == 'Batsman'].head(5)
bowlers = final_df[final_df['role'] == 'Bowler'].head(4)
all_rounders = final_df[final_df['role'] == 'All-Rounder'].head(2)

best_xi = pd.concat([batsmen, bowlers, all_rounders])

best_xi[['player', 'role', 'performance']]